# Stage 4: LLM-Based Biomedical Entity and Relation Extraction

This notebook implements the LLM-based extraction pipeline for the BioRED
dataset. It will generate structured biomedical entity and relation predictions
from title-and-abstract text, validate the returned output, and compare the
predictions against BioRED gold annotations using precision, recall, and
F1-score.

## Stage 4.1: Defining the Extraction Schema and Output Format

The purpose of this stage is to:

1. Define the BioRED entity categories.
2. Define the BioRED relation categories.
3. Design a strict JSON output format for the LLM.
4. Separate raw LLM output from post-processed evaluation output.
5. Validate the output structure before running extraction on the dataset.

In [1]:
BIORED_ENTITY_TYPES = [
    "CellLine",
    "ChemicalEntity",
    "DiseaseOrPhenotypicFeature",
    "GeneOrGeneProduct",
    "OrganismTaxon",
    "SequenceVariant"
]

print("Number of BioRED entity types:", len(BIORED_ENTITY_TYPES))
print(BIORED_ENTITY_TYPES)

Number of BioRED entity types: 6
['CellLine', 'ChemicalEntity', 'DiseaseOrPhenotypicFeature', 'GeneOrGeneProduct', 'OrganismTaxon', 'SequenceVariant']


In [2]:
BIORED_RELATION_TYPES = [
    "Association",
    "Bind",
    "Comparison",
    "Conversion",
    "Cotreatment",
    "Drug_Interaction",
    "Negative_Correlation",
    "Positive_Correlation"
]

print("Number of BioRED relation types:", len(BIORED_RELATION_TYPES))
print(BIORED_RELATION_TYPES)

Number of BioRED relation types: 8
['Association', 'Bind', 'Comparison', 'Conversion', 'Cotreatment', 'Drug_Interaction', 'Negative_Correlation', 'Positive_Correlation']


### Stage 4.1D: Selecting the LLM Output Format

The LLM will return structured JSON rather than free-text output.

The output will contain:

- `entities`: biomedical entities identified in the document
- `relations`: relations identified between those entities

Each entity will contain:
- a unique entity ID
- the exact entity text from the document
- one valid BioRED entity type

Each relation will contain:
- the ID of the first entity
- the ID of the second entity
- one valid BioRED relation type

Character offsets will not initially be generated by the LLM. They will be
recovered later using Python post-processing because exact character counting
may be unreliable in LLM-generated output.

In [3]:
LLM_OUTPUT_FORMAT = {
    "entities": [
        {
            "id": "E1",
            "text": "exact entity text from the document",
            "type": "BioRED entity type"
        }
    ],
    "relations": [
        {
            "head": "E1",
            "tail": "E2",
            "type": "BioRED relation type"
        }
    ]
}

LLM_OUTPUT_FORMAT

{'entities': [{'id': 'E1',
   'text': 'exact entity text from the document',
   'type': 'BioRED entity type'}],
 'relations': [{'head': 'E1', 'tail': 'E2', 'type': 'BioRED relation type'}]}

### Stage 4.1E: Character Offset Recovery Strategy

BioRED entity annotations contain character offsets identifying the exact
location of each entity mention in the title-and-abstract text.

The LLM will not be asked to generate character offsets directly because
LLMs may identify the correct entity but produce inaccurate character
positions.

Instead, the pipeline will use the following approach:

1. The LLM extracts the exact entity text and its BioRED entity type.
2. The LLM returns the result in structured JSON format.
3. Python searches for the extracted entity text in the original document.
4. Python calculates the start and end character offsets.
5. Ambiguous or repeated entity mentions are recorded for later validation.

This separates semantic extraction from deterministic character-offset
calculation and makes the pipeline easier to validate.

In [4]:
OFFSET_STRATEGY = {
    "llm_responsibility": [
        "Extract the exact entity text",
        "Assign a valid BioRED entity type",
        "Extract relations between identified entities"
    ],
    "python_responsibility": [
        "Locate the entity text in the original document",
        "Calculate start and end character offsets",
        "Detect repeated or ambiguous mentions",
        "Validate that the entity text appears in the document"
    ]
}

OFFSET_STRATEGY

{'llm_responsibility': ['Extract the exact entity text',
  'Assign a valid BioRED entity type',
  'Extract relations between identified entities'],
 'python_responsibility': ['Locate the entity text in the original document',
  'Calculate start and end character offsets',
  'Detect repeated or ambiguous mentions',
  'Validate that the entity text appears in the document']}

In [5]:
sample_text = (
    "Early-life exposure to inorganic arsenic is associated "
    "with an increased risk of cancer."
)

entity_text = "inorganic arsenic"

start_offset = sample_text.find(entity_text)
end_offset = start_offset + len(entity_text)

print("Entity text:", entity_text)
print("Start offset:", start_offset)
print("End offset:", end_offset)
print("Recovered text:", sample_text[start_offset:end_offset])

Entity text: inorganic arsenic
Start offset: 23
End offset: 40
Recovered text: inorganic arsenic


In [6]:
sample_text[start_offset:end_offset] == entity_text

True

In [7]:
print(
    "Offset validation passed:",
    sample_text[start_offset:end_offset] == entity_text
)

Offset validation passed: True


In [8]:
missing_entity = "arsenic exposure pathway"

missing_start = sample_text.find(missing_entity)

if missing_start == -1:
    print("Entity text was not found in the document.")
else:
    missing_end = missing_start + len(missing_entity)
    print("Start:", missing_start)
    print("End:", missing_end)

Entity text was not found in the document.


### Stage 4.1F: Handling Repeated Entity Mentions

The same biomedical entity may appear multiple times in one document.

Using Python's `find()` method returns only the first occurrence. Therefore,
the post-processing pipeline must identify all occurrences of an extracted
entity text.

For each entity, the pipeline will:

1. Search the complete title-and-abstract text.
2. Record every matching start and end offset.
3. Mark entities with multiple matches as ambiguous.
4. Resolve the correct mention during later post-processing or evaluation.

This prevents repeated entity mentions from being silently mapped only to
their first occurrence.

In [9]:
repeated_text = (
    "Arsenic exposure was investigated. "
    "Arsenic was associated with cancer."
)

entity_text = "Arsenic"

first_start = repeated_text.find(entity_text)
first_end = first_start + len(entity_text)

print("First start offset:", first_start)
print("First end offset:", first_end)
print("Recovered text:", repeated_text[first_start:first_end])

First start offset: 0
First end offset: 7
Recovered text: Arsenic


In [10]:
def find_all_occurrences(document_text, entity_text):
    """
    Find all exact occurrences of an entity text in a document.

    Returns a list of dictionaries containing start and end offsets.
    """

    occurrences = []
    search_start = 0

    while True:
        start_offset = document_text.find(entity_text, search_start)

        if start_offset == -1:
            break

        end_offset = start_offset + len(entity_text)

        occurrences.append({
            "start": start_offset,
            "end": end_offset
        })

        search_start = end_offset

    return occurrences

In [11]:
arsenic_occurrences = find_all_occurrences(
    repeated_text,
    entity_text
)

print("Number of occurrences:", len(arsenic_occurrences))
print(arsenic_occurrences)

Number of occurrences: 2
[{'start': 0, 'end': 7}, {'start': 35, 'end': 42}]


In [12]:
for occurrence_number, occurrence in enumerate(
    arsenic_occurrences,
    start=1
):
    start_offset = occurrence["start"]
    end_offset = occurrence["end"]

    recovered_text = repeated_text[start_offset:end_offset]

    print(
        f"Occurrence {occurrence_number}:",
        start_offset,
        end_offset,
        recovered_text
    )

Occurrence 1: 0 7 Arsenic
Occurrence 2: 35 42 Arsenic


In [13]:
if len(arsenic_occurrences) == 0:
    print("Entity text was not found.")
elif len(arsenic_occurrences) == 1:
    print("One exact entity occurrence was found.")
else:
    print(
        "Multiple entity occurrences were found. "
        "Mention resolution will be required."
    )

Multiple entity occurrences were found. Mention resolution will be required.


## Stage 4.2: Loading the BioRED Dataset

In this stage, the original BioRED dataset will be loaded into the new
LLM notebook.

The title and abstract of each BioRED document will later be reconstructed
and used as input for the LLM-based extraction pipeline.

The gold entity and relation annotations will be retained for evaluation.

In [14]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [15]:
import os

drive_root = "/content/drive/MyDrive"

all_json_files = []

for root, directories, files in os.walk(drive_root):
    for file_name in files:
        if file_name.lower().endswith(".json"):
            all_json_files.append(
                os.path.join(root, file_name)
            )

print("Total JSON files found:", len(all_json_files))

for file_path in all_json_files:
    print(file_path)

Total JSON files found: 8
/content/drive/MyDrive/Capstone_Project/Stage_4_Outputs/Pilot_Outputs/14510914_pilot_output.json
/content/drive/MyDrive/Capstone_Project/Stage_4_Outputs/Pilot_Outputs/14510914_pilot_output_with_offsets.json
/content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/PubMedBERT_RE_Weighted_Final/config.json
/content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/PubMedBERT_RE_Weighted_Final/tokenizer_config.json
/content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/PubMedBERT_RE_Weighted_Final/tokenizer.json
/content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/PubMedBERT_RE_Weighted_Final/relation_label_to_id.json
/content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/PubMedBERT_RE_Weighted_Final/relation_id_to_label.json
/content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/PubMedBERT_RE_Weighted_Final/final_test_results.json


In [16]:
!wget https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/BIORED.zip
!unzip -q BIORED.zip -d BioRED_data

--2026-08-03 16:54:09--  https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/BIORED.zip
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.7, 130.14.250.10, 2607:f220:41e:250::10, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2191684 (2.1M) [application/zip]
Saving to: ‘BIORED.zip’

BIORED.zip          100%[===================>]   2.09M  1.38MB/s    in 1.5s    

2026-08-03 16:54:13 (1.38 MB/s) - ‘BIORED.zip’ saved [2191684/2191684]



In [17]:
import json
from pathlib import Path

data_dir = Path("/content/BioRED_data/BioRED")

train_file = data_dir / "Train.BioC.JSON"
dev_file = data_dir / "Dev.BioC.JSON"
test_file = data_dir / "Test.BioC.JSON"


def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)


train_data = load_json(train_file)
dev_data = load_json(dev_file)
test_data = load_json(test_file)

print("BioRED datasets loaded successfully.")

BioRED datasets loaded successfully.


In [18]:
print("Training documents:", len(train_data["documents"]))
print("Development documents:", len(dev_data["documents"]))
print("Test documents:", len(test_data["documents"]))

print("\nTraining keys:", train_data.keys())
print(
    "First training document keys:",
    train_data["documents"][0].keys()
)

Training documents: 400
Development documents: 100
Test documents: 100

Training keys: dict_keys(['source', 'date', 'key', 'documents'])
First training document keys: dict_keys(['id', 'passages', 'relations'])


### Stage 4.2B: Reconstructing BioRED Title-and-Abstract Text

BioRED stores each document as multiple passages, typically a title and an
abstract.

For LLM-based extraction, these passages will be combined into one document-level
text while preserving their original character offsets.

The reconstructed text will be used as the input to the LLM, while the original
gold annotations will be retained for later evaluation.

In [19]:
first_document = train_data["documents"][0]

print("Document ID:", first_document["id"])
print("Number of passages:", len(first_document["passages"]))

for passage_index, passage in enumerate(first_document["passages"]):
    print(f"\nPassage {passage_index}")
    print("Offset:", passage.get("offset"))
    print("Type:", passage.get("infons", {}).get("type"))
    print("Text:", passage.get("text"))

Document ID: 10491763
Number of passages: 2

Passage 0
Offset: 0
Type: None
Text: Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and between genetic variability and estimates of insulin secretion.

Passage 1
Offset: 159
Type: None
Text: The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 54 patients with late-onset Type II diabetes by combined single strand conformational polymorphism-heteroduplex analysis follow

In [20]:
def reconstruct_document_text(document):
    """
    Reconstruct complete BioRED document text using passage offsets.

    Spaces are inserted where needed so that the reconstructed text preserves
    the original BioRED character positions.
    """

    passages = sorted(
        document.get("passages", []),
        key=lambda passage: passage.get("offset", 0)
    )

    reconstructed_text = ""

    for passage in passages:
        passage_offset = passage.get("offset", 0)
        passage_text = passage.get("text", "")

        if len(reconstructed_text) < passage_offset:
            missing_space = passage_offset - len(reconstructed_text)
            reconstructed_text += " " * missing_space

        reconstructed_text += passage_text

    return reconstructed_text

In [21]:
first_document_text = reconstruct_document_text(first_document)

print("Document ID:", first_document["id"])
print("Reconstructed text length:", len(first_document_text))
print("\nReconstructed title-and-abstract text:\n")
print(first_document_text)

Document ID: 10491763
Reconstructed text length: 1797

Reconstructed title-and-abstract text:

Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and between genetic variability and estimates of insulin secretion. The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 54 patients with late-onset Type II diabetes by combined single strand conformational polymorphism-heteroduplex analysis followed by direct sequencing of 

In [22]:
passage_validation_errors = []

for passage in first_document["passages"]:
    passage_offset = passage["offset"]
    passage_text = passage["text"]
    passage_end = passage_offset + len(passage_text)

    recovered_passage = first_document_text[
        passage_offset:passage_end
    ]

    if recovered_passage != passage_text:
        passage_validation_errors.append({
            "offset": passage_offset,
            "expected": passage_text,
            "recovered": recovered_passage
        })

print("Passage validation errors:", len(passage_validation_errors))

Passage validation errors: 0


### Stage 4.2C: Inspecting Gold Entities and Relations

The BioRED document contains gold-standard entity annotations inside its
passages and gold relation annotations at the document level.

These annotations will later be used as the reference labels for evaluating
the LLM predictions.

In [23]:
gold_entities = []

for passage in first_document["passages"]:
    passage_type = passage.get("infons", {}).get("type")

    for annotation in passage.get("annotations", []):
        annotation_info = annotation.get("infons", {})
        locations = annotation.get("locations", [])

        for location in locations:
            entity_start = location["offset"]
            entity_length = location["length"]
            entity_end = entity_start + entity_length

            gold_entities.append({
                "annotation_id": annotation.get("id"),
                "identifier": annotation_info.get("identifier"),
                "type": annotation_info.get("type"),
                "text": annotation.get("text"),
                "start": entity_start,
                "end": entity_end,
                "passage_type": passage_type
            })

print("Number of gold entities:", len(gold_entities))

Number of gold entities: 32


In [24]:
for entity in gold_entities[:10]:
    print(entity)

{'annotation_id': '0', 'identifier': '3175', 'type': 'GeneOrGeneProduct', 'text': 'Hepatocyte nuclear factor-6', 'start': 0, 'end': 27, 'passage_type': None}
{'annotation_id': '1', 'identifier': 'D003924', 'type': 'DiseaseOrPhenotypicFeature', 'text': 'type II diabetes', 'start': 74, 'end': 90, 'passage_type': None}
{'annotation_id': '2', 'identifier': '3630', 'type': 'GeneOrGeneProduct', 'text': 'insulin', 'start': 140, 'end': 147, 'passage_type': None}
{'annotation_id': '3', 'identifier': '3175', 'type': 'GeneOrGeneProduct', 'text': 'hepatocyte nuclear factor (HNF)-6', 'start': 184, 'end': 217, 'passage_type': None}
{'annotation_id': '4', 'identifier': 'D003924', 'type': 'DiseaseOrPhenotypicFeature', 'text': 'maturity-onset diabetes', 'start': 292, 'end': 315, 'passage_type': None}
{'annotation_id': '5', 'identifier': '3175', 'type': 'GeneOrGeneProduct', 'text': 'HNF-6', 'start': 389, 'end': 394, 'passage_type': None}
{'annotation_id': '6', 'identifier': 'D003924', 'type': 'DiseaseOr

In [25]:
entity_offset_errors = []

for entity in gold_entities:
    recovered_text = first_document_text[
        entity["start"]:entity["end"]
    ]

    if recovered_text != entity["text"]:
        entity_offset_errors.append({
            "annotation_id": entity["annotation_id"],
            "expected": entity["text"],
            "recovered": recovered_text,
            "start": entity["start"],
            "end": entity["end"]
        })

print("Entity offset validation errors:", len(entity_offset_errors))

Entity offset validation errors: 0


In [26]:
gold_relations = []

for relation in first_document.get("relations", []):
    relation_info = relation.get("infons", {})
    relation_nodes = relation.get("nodes", [])

    relation_record = {
        "relation_id": relation.get("id"),
        "type": relation_info.get("type"),
        "nodes": []
    }

    for node in relation_nodes:
        relation_record["nodes"].append({
            "refid": node.get("refid"),
            "role": node.get("role")
        })

    gold_relations.append(relation_record)

print("Number of gold relations:", len(gold_relations))

Number of gold relations: 3


In [27]:
for relation in gold_relations[:10]:
    print(relation)

{'relation_id': 'R0', 'type': 'Association', 'nodes': []}
{'relation_id': 'R1', 'type': 'Positive_Correlation', 'nodes': []}
{'relation_id': 'R2', 'type': 'Association', 'nodes': []}


In [28]:
print("Document ID:", first_document["id"])
print("Document length:", len(first_document_text))
print("Gold entities:", len(gold_entities))
print("Gold relations:", len(gold_relations))
print("Entity offset errors:", len(entity_offset_errors))

Document ID: 10491763
Document length: 1797
Gold entities: 32
Gold relations: 3
Entity offset errors: 0


### Stage 4.2D: Creating a Reusable Document Preparation Function

A reusable function will now combine the main preparation steps for any
BioRED document.

For each document, the function will return:

- the document ID
- the reconstructed title-and-abstract text
- the gold entity annotations
- the gold relation annotations
- the number of offset-validation errors

This structured output will later support LLM prompting and evaluation.

In [29]:
def prepare_biored_document(document):
    """
    Prepare one BioRED document for the LLM extraction pipeline.

    Returns
    -------
    dict
        A dictionary containing:
        - document_id
        - text
        - gold_entities
        - gold_relations
        - entity_offset_errors
    """

    document_text = reconstruct_document_text(document)

    gold_entities = []

    for passage in document.get("passages", []):
        passage_type = passage.get("infons", {}).get("type")

        for annotation in passage.get("annotations", []):
            annotation_info = annotation.get("infons", {})
            locations = annotation.get("locations", [])

            for location in locations:
                entity_start = location["offset"]
                entity_length = location["length"]
                entity_end = entity_start + entity_length

                gold_entities.append({
                    "annotation_id": annotation.get("id"),
                    "identifier": annotation_info.get("identifier"),
                    "type": annotation_info.get("type"),
                    "text": annotation.get("text"),
                    "start": entity_start,
                    "end": entity_end,
                    "passage_type": passage_type
                })

    entity_offset_errors = []

    for entity in gold_entities:
        recovered_text = document_text[
            entity["start"]:entity["end"]
        ]

        if recovered_text != entity["text"]:
            entity_offset_errors.append({
                "annotation_id": entity["annotation_id"],
                "expected": entity["text"],
                "recovered": recovered_text,
                "start": entity["start"],
                "end": entity["end"]
            })

    gold_relations = []

    for relation in document.get("relations", []):
        relation_info = relation.get("infons", {})
        relation_nodes = relation.get("nodes", [])

        gold_relations.append({
            "relation_id": relation.get("id"),
            "type": relation_info.get("type"),
            "nodes": [
                {
                    "refid": node.get("refid"),
                    "role": node.get("role")
                }
                for node in relation_nodes
            ]
        })

    return {
        "document_id": document.get("id"),
        "text": document_text,
        "gold_entities": gold_entities,
        "gold_relations": gold_relations,
        "entity_offset_errors": entity_offset_errors
    }

In [30]:
prepared_first_document = prepare_biored_document(
    train_data["documents"][0]
)

print("Document ID:", prepared_first_document["document_id"])
print("Text length:", len(prepared_first_document["text"]))
print("Gold entities:", len(prepared_first_document["gold_entities"]))
print("Gold relations:", len(prepared_first_document["gold_relations"]))
print(
    "Entity offset errors:",
    len(prepared_first_document["entity_offset_errors"])
)

Document ID: 10491763
Text length: 1797
Gold entities: 32
Gold relations: 3
Entity offset errors: 0


In [31]:
print(prepared_first_document.keys())

dict_keys(['document_id', 'text', 'gold_entities', 'gold_relations', 'entity_offset_errors'])


### Stage 4.2E: Preparing All BioRED Documents

The reusable preparation function will now be applied to all BioRED splits:

- 400 training documents
- 100 development documents
- 100 test documents

This creates a consistent document-level structure containing reconstructed
text, gold entities, gold relations, and offset-validation results for every
BioRED document.

In [32]:
prepared_train_documents = [
    prepare_biored_document(document)
    for document in train_data["documents"]
]

prepared_dev_documents = [
    prepare_biored_document(document)
    for document in dev_data["documents"]
]

prepared_test_documents = [
    prepare_biored_document(document)
    for document in test_data["documents"]
]

In [33]:
print("Prepared training documents:", len(prepared_train_documents))
print("Prepared development documents:", len(prepared_dev_documents))
print("Prepared test documents:", len(prepared_test_documents))

total_prepared_documents = (
    len(prepared_train_documents)
    + len(prepared_dev_documents)
    + len(prepared_test_documents)
)

print("Total prepared documents:", total_prepared_documents)

Prepared training documents: 400
Prepared development documents: 100
Prepared test documents: 100
Total prepared documents: 600


In [34]:
train_offset_errors = sum(
    len(document["entity_offset_errors"])
    for document in prepared_train_documents
)

dev_offset_errors = sum(
    len(document["entity_offset_errors"])
    for document in prepared_dev_documents
)

test_offset_errors = sum(
    len(document["entity_offset_errors"])
    for document in prepared_test_documents
)

print("Training offset errors:", train_offset_errors)
print("Development offset errors:", dev_offset_errors)
print("Test offset errors:", test_offset_errors)
print(
    "Total offset errors:",
    train_offset_errors + dev_offset_errors + test_offset_errors
)

Training offset errors: 0
Development offset errors: 0
Test offset errors: 0
Total offset errors: 0


In [35]:
total_train_entities = sum(
    len(document["gold_entities"])
    for document in prepared_train_documents
)

total_dev_entities = sum(
    len(document["gold_entities"])
    for document in prepared_dev_documents
)

total_test_entities = sum(
    len(document["gold_entities"])
    for document in prepared_test_documents
)

total_train_relations = sum(
    len(document["gold_relations"])
    for document in prepared_train_documents
)

total_dev_relations = sum(
    len(document["gold_relations"])
    for document in prepared_dev_documents
)

total_test_relations = sum(
    len(document["gold_relations"])
    for document in prepared_test_documents
)

print("Training entities:", total_train_entities)
print("Development entities:", total_dev_entities)
print("Test entities:", total_test_entities)

print("\nTraining relations:", total_train_relations)
print("Development relations:", total_dev_relations)
print("Test relations:", total_test_relations)

Training entities: 13351
Development entities: 3533
Test entities: 3535

Training relations: 4178
Development relations: 1162
Test relations: 1163


### Stage 4.2F: Creating a Small Prompt-Development Sample

A small subset of development documents will be used to test the LLM prompt,
JSON output format, validation logic, and error handling before processing the
full BioRED dataset.

The final dataset remains unchanged. This sample is only used for pipeline
testing.

In [36]:
prompt_development_sample = prepared_dev_documents[:3]

print("Prompt-development documents:", len(prompt_development_sample))

for document in prompt_development_sample:
    print(
        "Document ID:",
        document["document_id"],
        "| Text length:",
        len(document["text"]),
        "| Gold entities:",
        len(document["gold_entities"]),
        "| Gold relations:",
        len(document["gold_relations"])
    )

Prompt-development documents: 3
Document ID: 14510914 | Text length: 1920 | Gold entities: 37 | Gold relations: 12
Document ID: 15096016 | Text length: 1362 | Gold entities: 14 | Gold relations: 1
Document ID: 16152606 | Text length: 1802 | Gold entities: 40 | Gold relations: 10


In [37]:
sample_document = prompt_development_sample[0]

print("Document ID:", sample_document["document_id"])
print("\nDocument text:\n")
print(sample_document["text"])

Document ID: 14510914

Document text:

Congenital hypothyroidism due to a new deletion in the sodium/iodide symporter protein. OBJECTIVE: Iodide transport defect (ITD) is a rare disorder characterised by an inability of the thyroid to maintain an iodide gradient across the basolateral membrane of thyroid follicular cells, that often results in congenital hypothyroidism. When present the defect is also found in the salivary glands and gastric mucosa and it has been shown to arise from abnormalities of the sodium/iodide symporter (NIS). PATIENT: We describe a woman with hypothyroidism identified at the 3rd month of life. The diagnosis of ITD was suspected because of nodular goitre, and little if any iodide uptake by the thyroid and salivary glands. Treatment with iodide partially corrected the hypothyroidism; however, long-term substitution therapy with L-thyroxine was started. MEASUREMENTS: Thyroid radioiodide uptake was only 1.4% and 0.3% at 1 and 24 h after the administration of recom

In [38]:
print(sample_document.keys())

dict_keys(['document_id', 'text', 'gold_entities', 'gold_relations', 'entity_offset_errors'])


## Stage 4.3: Constructing the LLM Extraction Prompt

The LLM will receive the reconstructed BioRED title-and-abstract text and will
be instructed to extract biomedical entities and relations using only the
BioRED schema.

The first experiment will use a zero-shot prompt. No gold annotations from the
document will be shown to the LLM.

The prompt will require strict JSON output so that the response can be parsed
and evaluated automatically.

### Stage 4.3A: Creating a Zero-Shot Prompt Template

The prompt will contain:

1. The extraction task.
2. The six permitted BioRED entity types.
3. The eight permitted BioRED relation types.
4. Rules for copying exact entity text.
5. The required JSON output structure.
6. The document text.

In [39]:
import json


def build_zero_shot_prompt(document_text):
    """
    Build a zero-shot prompt for BioRED entity and relation extraction.

    Parameters
    ----------
    document_text : str
        Reconstructed BioRED title-and-abstract text.

    Returns
    -------
    str
        Complete prompt to send to the LLM.
    """

    entity_types = [
        "CellLine",
        "ChemicalEntity",
        "DiseaseOrPhenotypicFeature",
        "GeneOrGeneProduct",
        "OrganismTaxon",
        "SequenceVariant"
    ]

    relation_types = [
        "Association",
        "Bind",
        "Comparison",
        "Conversion",
        "Cotreatment",
        "Drug_Interaction",
        "Negative_Correlation",
        "Positive_Correlation"
    ]

    output_example = {
        "entities": [
            {
                "id": "E1",
                "text": "exact entity text from the document",
                "type": "one valid BioRED entity type"
            }
        ],
        "relations": [
            {
                "head": "E1",
                "tail": "E2",
                "type": "one valid BioRED relation type"
            }
        ]
    }

    prompt = f"""
You are a biomedical information extraction system.

Extract all explicitly mentioned biomedical entities and relations from the
document below using the BioRED schema.

PERMITTED ENTITY TYPES:
{json.dumps(entity_types, indent=2)}

PERMITTED RELATION TYPES:
{json.dumps(relation_types, indent=2)}

ENTITY RULES:
1. Extract only entities explicitly present in the document.
2. Copy each entity text exactly as it appears in the document.
3. Do not paraphrase, normalise, expand, or correct entity text.
4. Assign every entity a unique ID beginning with E1, E2, E3, and so on.
5. Use only the permitted BioRED entity types.
6. Do not create duplicate entity records for the same textual mention.
7. Include repeated mentions only when they represent separate mentions that
   are needed for relation extraction.

RELATION RULES:
1. Extract only relations explicitly supported by the document.
2. Use only the permitted BioRED relation types.
3. The head and tail fields must reference valid entity IDs.
4. Do not create relations between entities that were not extracted.
5. Do not connect an entity to itself.
6. Do not invent relations from general biomedical knowledge.

OUTPUT RULES:
1. Return one valid JSON object only.
2. Do not include Markdown code fences.
3. Do not include explanations before or after the JSON.
4. Use exactly the keys shown in the required output structure.
5. If no entities or relations are found, return empty lists.

REQUIRED OUTPUT STRUCTURE:
{json.dumps(output_example, indent=2)}

DOCUMENT:
{document_text}
""".strip()

    return prompt

In [40]:
first_prompt = build_zero_shot_prompt(
    prompt_development_sample[0]["text"]
)

print(first_prompt)

You are a biomedical information extraction system.

Extract all explicitly mentioned biomedical entities and relations from the
document below using the BioRED schema.

PERMITTED ENTITY TYPES:
[
  "CellLine",
  "ChemicalEntity",
  "DiseaseOrPhenotypicFeature",
  "GeneOrGeneProduct",
  "OrganismTaxon",
  "SequenceVariant"
]

PERMITTED RELATION TYPES:
[
  "Association",
  "Bind",
  "Comparison",
  "Conversion",
  "Cotreatment",
  "Drug_Interaction",
  "Negative_Correlation",
  "Positive_Correlation"
]

ENTITY RULES:
1. Extract only entities explicitly present in the document.
2. Copy each entity text exactly as it appears in the document.
3. Do not paraphrase, normalise, expand, or correct entity text.
4. Assign every entity a unique ID beginning with E1, E2, E3, and so on.
5. Use only the permitted BioRED entity types.
6. Do not create duplicate entity records for the same textual mention.
7. Include repeated mentions only when they represent separate mentions that
   are needed for re

In [41]:
print("Prompt characters:", len(first_prompt))
print("Document characters:", len(prompt_development_sample[0]["text"]))

Prompt characters: 3888
Document characters: 1920


### Stage 4.3B: Validating the Prompt Structure

Before connecting an LLM, the prompt will be checked automatically to confirm
that it contains all required BioRED labels, output instructions, and the
complete document text.

This prevents incomplete or malformed prompts from being sent during the
extraction experiment.

In [42]:
def validate_extraction_prompt(prompt, document_text):
    """
    Validate that the BioRED extraction prompt contains all required parts.

    Returns
    -------
    is_valid : bool
        True when all checks pass.
    checks : dict
        Individual validation results.
    """

    required_entity_types = [
        "CellLine",
        "ChemicalEntity",
        "DiseaseOrPhenotypicFeature",
        "GeneOrGeneProduct",
        "OrganismTaxon",
        "SequenceVariant"
    ]

    required_relation_types = [
        "Association",
        "Bind",
        "Comparison",
        "Conversion",
        "Cotreatment",
        "Drug_Interaction",
        "Negative_Correlation",
        "Positive_Correlation"
    ]

    checks = {
        "prompt_is_string": isinstance(prompt, str),
        "prompt_not_empty": isinstance(prompt, str) and bool(prompt.strip()),
        "document_text_present": document_text in prompt,
        "entities_key_present": '"entities"' in prompt,
        "relations_key_present": '"relations"' in prompt,
        "head_key_present": '"head"' in prompt,
        "tail_key_present": '"tail"' in prompt,
        "json_only_instruction_present": (
            "Return one valid JSON object only." in prompt
        ),
        "all_entity_types_present": all(
            entity_type in prompt
            for entity_type in required_entity_types
        ),
        "all_relation_types_present": all(
            relation_type in prompt
            for relation_type in required_relation_types
        )
    }

    is_valid = all(checks.values())

    return is_valid, checks

In [43]:
prompt_is_valid, prompt_checks = validate_extraction_prompt(
    first_prompt,
    prompt_development_sample[0]["text"]
)

print("Prompt is valid:", prompt_is_valid)

for check_name, check_result in prompt_checks.items():
    print(f"{check_name}: {check_result}")

Prompt is valid: True
prompt_is_string: True
prompt_not_empty: True
document_text_present: True
entities_key_present: True
relations_key_present: True
head_key_present: True
tail_key_present: True
json_only_instruction_present: True
all_entity_types_present: True
all_relation_types_present: True


In [44]:
pilot_prompt_validation_results = []

for document in prompt_development_sample:
    prompt = build_zero_shot_prompt(document["text"])

    is_valid, checks = validate_extraction_prompt(
        prompt,
        document["text"]
    )

    pilot_prompt_validation_results.append({
        "document_id": document["document_id"],
        "prompt_length": len(prompt),
        "is_valid": is_valid,
        "failed_checks": [
            check_name
            for check_name, passed in checks.items()
            if not passed
        ]
    })

for result in pilot_prompt_validation_results:
    print(result)

{'document_id': '14510914', 'prompt_length': 3888, 'is_valid': True, 'failed_checks': []}
{'document_id': '15096016', 'prompt_length': 3330, 'is_valid': True, 'failed_checks': []}
{'document_id': '16152606', 'prompt_length': 3770, 'is_valid': True, 'failed_checks': []}


### Stage 4.3C: Selecting an Open-Source LLM

The LLM-based extraction pipeline will use the open-weight
Qwen2.5-7B-Instruct model through Hugging Face Transformers.

The model will run locally in Google Colab using 4-bit quantisation to reduce
GPU memory requirements.

This approach avoids paid API usage and allows the extraction experiment to be
reproduced without relying on a proprietary API.

The initial experiment will run on three development documents. After prompt,
generation, JSON parsing, and validation are confirmed, the model will be
applied to the larger BioRED dataset.

In [45]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.6 MB/s eta 0:00:00


In [46]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [47]:
LLM_CONFIG = {
    "provider": "Hugging Face",
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "execution": "Local Colab GPU",
    "quantisation": "4-bit",
    "prompt_method": "zero-shot",
    "structured_output": "Prompt-constrained JSON",
    "api_required": False
}

LLM_CONFIG

{'provider': 'Hugging Face',
 'model': 'Qwen/Qwen2.5-7B-Instruct',
 'execution': 'Local Colab GPU',
 'quantisation': '4-bit',
 'prompt_method': 'zero-shot',
 'structured_output': 'Prompt-constrained JSON',
 'api_required': False}

## Stage 4.4: Loading and Running the Open-Source LLM

### Stage 4.4A: Loading Qwen2.5-7B-Instruct

Qwen2.5-7B-Instruct will be loaded from Hugging Face using 4-bit
quantisation.

Four-bit quantisation reduces GPU memory usage and makes it possible to run
the seven-billion-parameter model on a Google Colab GPU.

At this stage, the tokenizer and model will only be loaded and validated.
No BioRED extraction will be performed yet.

In [48]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [49]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. Change the Colab runtime type to a GPU."
    )

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [50]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

In [51]:
MODEL_NAME = LLM_CONFIG["model"]

print("Selected model:", MODEL_NAME)

Selected model: Qwen/Qwen2.5-7B-Instruct


In [52]:
quantisation_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(quantisation_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [53]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")
print("Vocabulary size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("Pad token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully.
Vocabulary size: 151665
EOS token: <|im_end|>
Pad token: <|endoftext|>


In [54]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantisation_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded successfully.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.


## Stage 4.4B: First Pilot Biomedical Extraction

In this stage, the loaded Qwen2.5-7B-Instruct model will be tested on one
BioRED development document.

The model will receive the document title and abstract through the validated
zero-shot prompt and will be asked to jointly perform:

- Named Entity Recognition
- Relation Extraction
- Structured JSON generation

The purpose of this pilot run is to inspect the model’s raw response before
processing the full development sample.

The generated response will later be:

1. decoded from model tokens;
2. parsed into JSON;
3. validated against the BioRED entity and relation schema;
4. checked for hallucinated or missing entity mentions;
5. prepared for character-offset recovery.

No prompt changes will be made based on the final evaluation set.


In [55]:
print("Model class:", model.__class__.__name__)
print("Model device:", model.device)
print("Training mode:", model.training)
print("4-bit model:", getattr(model, "is_loaded_in_4bit", False))

allocated_memory_gb = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_memory_gb = torch.cuda.memory_reserved() / (1024 ** 3)

print(f"Allocated GPU memory: {allocated_memory_gb:.2f} GB")
print(f"Reserved GPU memory: {reserved_memory_gb:.2f} GB")

Model class: Qwen2ForCausalLM
Model device: cuda:0
Training mode: False
4-bit model: True
Allocated GPU memory: 5.18 GB
Reserved GPU memory: 5.33 GB


In [56]:
pilot_document = prompt_development_sample[0]

print("Document ID:", pilot_document["document_id"])
print("Text length:", len(pilot_document["text"]))
print()
print(pilot_document["text"][:1000])

Document ID: 14510914
Text length: 1920

Congenital hypothyroidism due to a new deletion in the sodium/iodide symporter protein. OBJECTIVE: Iodide transport defect (ITD) is a rare disorder characterised by an inability of the thyroid to maintain an iodide gradient across the basolateral membrane of thyroid follicular cells, that often results in congenital hypothyroidism. When present the defect is also found in the salivary glands and gastric mucosa and it has been shown to arise from abnormalities of the sodium/iodide symporter (NIS). PATIENT: We describe a woman with hypothyroidism identified at the 3rd month of life. The diagnosis of ITD was suspected because of nodular goitre, and little if any iodide uptake by the thyroid and salivary glands. Treatment with iodide partially corrected the hypothyroidism; however, long-term substitution therapy with L-thyroxine was started. MEASUREMENTS: Thyroid radioiodide uptake was only 1.4% and 0.3% at 1 and 24 h after the administration of rec

In [57]:
pilot_prompt = build_zero_shot_prompt(
    document_text=pilot_document["text"]
)

print("Prompt length:", len(pilot_prompt))
print()
print(pilot_prompt[:2000])

Prompt length: 3888

You are a biomedical information extraction system.

Extract all explicitly mentioned biomedical entities and relations from the
document below using the BioRED schema.

PERMITTED ENTITY TYPES:
[
  "CellLine",
  "ChemicalEntity",
  "DiseaseOrPhenotypicFeature",
  "GeneOrGeneProduct",
  "OrganismTaxon",
  "SequenceVariant"
]

PERMITTED RELATION TYPES:
[
  "Association",
  "Bind",
  "Comparison",
  "Conversion",
  "Cotreatment",
  "Drug_Interaction",
  "Negative_Correlation",
  "Positive_Correlation"
]

ENTITY RULES:
1. Extract only entities explicitly present in the document.
2. Copy each entity text exactly as it appears in the document.
3. Do not paraphrase, normalise, expand, or correct entity text.
4. Assign every entity a unique ID beginning with E1, E2, E3, and so on.
5. Use only the permitted BioRED entity types.
6. Do not create duplicate entity records for the same textual mention.
7. Include repeated mentions only when they represent separate mentions that

In [58]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a biomedical information extraction system. "
            "Follow the user's instructions exactly and return valid JSON only."
        )
    },
    {
        "role": "user",
        "content": pilot_prompt
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("Formatted prompt length:", len(formatted_prompt))

Formatted prompt length: 4086


In [59]:
model_inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(model.device)

input_token_count = model_inputs["input_ids"].shape[1]

print("Input token count:", input_token_count)

Input token count: 968


In [60]:
import torch

with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1200,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id
    )

new_token_ids = generated_ids[
    :, model_inputs["input_ids"].shape[1]:
]

raw_response = tokenizer.batch_decode(
    new_token_ids,
    skip_special_tokens=True
)[0]

print("Generated token count:", new_token_ids.shape[1])
print()
print(raw_response)

Generated token count: 738

{
  "entities": [
    {
      "id": "E1",
      "text": "Congenital hypothyroidism",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
      "id": "E2",
      "text": "sodium/iodide symporter protein",
      "type": "GeneOrGeneProduct"
    },
    {
      "id": "E3",
      "text": "iodide transport defect",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
      "id": "E4",
      "text": "thyroid",
      "type": "OrganismTaxon"
    },
    {
      "id": "E5",
      "text": "salivary glands",
      "type": "OrganismTaxon"
    },
    {
      "id": "E6",
      "text": "gastric mucosa",
      "type": "OrganismTaxon"
    },
    {
      "id": "E7",
      "text": "sodium/iodide symporter (NIS)",
      "type": "GeneOrGeneProduct"
    },
    {
      "id": "E8",
      "text": "hypothyroidism",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
      "id": "E9",
      "text": "nodular goitre",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
     

In [61]:
import json

try:
    parsed_response = json.loads(raw_response)
    json_valid = True
    json_error = None

    print("JSON parsing successful.")
    print("Top-level keys:", list(parsed_response.keys()))
    print("Predicted entities:", len(parsed_response["entities"]))
    print("Predicted relations:", len(parsed_response["relations"]))

except json.JSONDecodeError as error:
    parsed_response = None
    json_valid = False
    json_error = str(error)

    print("JSON parsing failed.")
    print("Error:", json_error)

JSON parsing successful.
Top-level keys: ['entities', 'relations']
Predicted entities: 17
Predicted relations: 3


In [62]:
import json
from pathlib import Path

pilot_output_dir = Path(
    "/content/drive/MyDrive/Capstone_Project/Stage_4_Outputs/Pilot_Outputs"
)
pilot_output_dir.mkdir(parents=True, exist_ok=True)

pilot_result = {
    "document_id": pilot_document["document_id"],
    "model": MODEL_NAME,
    "prompt_method": "zero-shot",
    "input_token_count": input_token_count,
    "generated_token_count": new_token_ids.shape[1],
    "raw_response": raw_response,
    "json_valid": json_valid,
    "json_error": json_error,
    "parsed_response": parsed_response
}

pilot_output_path = (
    pilot_output_dir /
    f'{pilot_document["document_id"]}_pilot_output.json'
)

with open(pilot_output_path, "w", encoding="utf-8") as file:
    json.dump(
        pilot_result,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Pilot output saved to:")
print(pilot_output_path)

Pilot output saved to:
/content/drive/MyDrive/Capstone_Project/Stage_4_Outputs/Pilot_Outputs/14510914_pilot_output.json


### Stage 4.4C: Schema and Text Validation

This stage validates the parsed Qwen output before evaluation.

The validation checks whether:

- the response contains the required `entities` and `relations` lists;
- all predicted labels belong to the BioRED schema;
- entity IDs are present and unique;
- each predicted entity text appears exactly in the source document;
- relation head and tail IDs reference valid predicted entities;
- no relation connects an entity to itself.

This stage checks the structural validity of the LLM output. It does not yet
measure whether the predictions match the BioRED gold annotations.

In [63]:
def validate_llm_extraction(
    parsed_output,
    document_text,
    permitted_entity_types,
    permitted_relation_types
):
    errors = []
    warnings = []

    if not isinstance(parsed_output, dict):
        return {
            "valid": False,
            "errors": ["Output is not a dictionary."],
            "warnings": [],
            "entity_results": [],
            "relation_results": []
        }

    entities = parsed_output.get("entities")
    relations = parsed_output.get("relations")

    if not isinstance(entities, list):
        errors.append("'entities' must be a list.")
        entities = []

    if not isinstance(relations, list):
        errors.append("'relations' must be a list.")
        relations = []

    entity_ids = []
    entity_results = []

    for index, entity in enumerate(entities):
        if not isinstance(entity, dict):
            errors.append(f"Entity {index} is not a dictionary.")
            continue

        entity_id = entity.get("id")
        entity_text = entity.get("text")
        entity_type = entity.get("type")

        # Validate entity ID
        if not isinstance(entity_id, str) or not entity_id.strip():
            errors.append(f"Entity {index} has an invalid ID.")
        else:
            entity_ids.append(entity_id)

        # Validate entity text
        exact_text_found = False
        case_insensitive_found = False

        if not isinstance(entity_text, str) or not entity_text.strip():
            errors.append(
                f"Entity {entity_id or index} has invalid text."
            )
        else:
            exact_text_found = entity_text in document_text
            case_insensitive_found = (
                entity_text.lower() in document_text.lower()
            )

            if not exact_text_found:
                if case_insensitive_found:
                    warnings.append(
                        f"Entity {entity_id}: text found only with "
                        f"different capitalisation: {entity_text!r}"
                    )
                else:
                    errors.append(
                        f"Entity {entity_id}: text not found in the "
                        f"document: {entity_text!r}"
                    )

        # Validate entity type
        if entity_type not in permitted_entity_types:
            errors.append(
                f"Entity {entity_id or index} has invalid type: "
                f"{entity_type!r}"
            )

        entity_results.append({
            "id": entity_id,
            "text": entity_text,
            "type": entity_type,
            "exact_text_found": exact_text_found,
            "case_insensitive_found": case_insensitive_found
        })

    # Check duplicate entity IDs
    duplicate_ids = sorted({
        entity_id
        for entity_id in entity_ids
        if entity_ids.count(entity_id) > 1
    })

    if duplicate_ids:
        errors.append(
            f"Duplicate entity IDs found: {duplicate_ids}"
        )

    valid_entity_ids = set(entity_ids)
    relation_results = []

    for index, relation in enumerate(relations):
        if not isinstance(relation, dict):
            errors.append(f"Relation {index} is not a dictionary.")
            continue

        head = relation.get("head")
        tail = relation.get("tail")
        relation_type = relation.get("type")

        head_valid = head in valid_entity_ids
        tail_valid = tail in valid_entity_ids
        self_relation = head == tail

        if not head_valid:
            errors.append(
                f"Relation {index} has an invalid head ID: {head!r}"
            )

        if not tail_valid:
            errors.append(
                f"Relation {index} has an invalid tail ID: {tail!r}"
            )

        if self_relation:
            errors.append(
                f"Relation {index} connects entity {head!r} to itself."
            )

        if relation_type not in permitted_relation_types:
            errors.append(
                f"Relation {index} has invalid type: "
                f"{relation_type!r}"
            )

        relation_results.append({
            "head": head,
            "tail": tail,
            "type": relation_type,
            "head_valid": head_valid,
            "tail_valid": tail_valid,
            "self_relation": self_relation
        })

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
        "entity_results": entity_results,
        "relation_results": relation_results
    }

In [64]:
pilot_validation = validate_llm_extraction(
    parsed_output=parsed_response,
    document_text=pilot_document["text"],
    permitted_entity_types=BIORED_ENTITY_TYPES,
    permitted_relation_types=BIORED_RELATION_TYPES
)

print("Structurally valid:", pilot_validation["valid"])
print("Number of errors:", len(pilot_validation["errors"]))
print("Number of warnings:", len(pilot_validation["warnings"]))

if pilot_validation["errors"]:
    print("\nValidation errors:")
    for error in pilot_validation["errors"]:
        print("-", error)

if pilot_validation["warnings"]:
    print("\nValidation warnings:")
    for warning in pilot_validation["warnings"]:
        print("-", warning)

Structurally valid: True
Number of errors: 0
Number of warnings: 2

Validation warnings:
- Entity E3: text found only with different capitalisation: 'iodide transport defect'
- Entity E11: text found only with different capitalisation: 'thyroid radioiodide uptake'


### Stage 4.4D: Character Offset Recovery

This stage recovers character-level start and end offsets for each predicted
entity by searching the original BioRED document text.

Exact case-sensitive matches are preferred. When an entity is found only with
different capitalisation, the original document span is recovered but the
case mismatch is recorded.

If the same entity text occurs more than once, all candidate offsets are
retained and the entity is marked as ambiguous rather than assigning an
arbitrary occurrence.

In [65]:
def find_all_text_occurrences(
    document_text,
    entity_text,
    case_sensitive=True
):
    if not isinstance(entity_text, str) or not entity_text:
        return []

    search_document = (
        document_text
        if case_sensitive
        else document_text.lower()
    )

    search_entity = (
        entity_text
        if case_sensitive
        else entity_text.lower()
    )

    occurrences = []
    search_start = 0

    while True:
        start = search_document.find(
            search_entity,
            search_start
        )

        if start == -1:
            break

        end = start + len(entity_text)

        occurrences.append({
            "start": start,
            "end": end,
            "document_text": document_text[start:end]
        })

        search_start = start + 1

    return occurrences

In [66]:
def recover_entity_offsets(
    parsed_output,
    document_text
):
    recovered_entities = []
    summary = {
        "unique_exact_matches": 0,
        "multiple_exact_matches": 0,
        "unique_case_insensitive_matches": 0,
        "multiple_case_insensitive_matches": 0,
        "not_found": 0
    }

    for entity in parsed_output.get("entities", []):
        recovered_entity = entity.copy()
        entity_text = entity.get("text", "")

        exact_occurrences = find_all_text_occurrences(
            document_text=document_text,
            entity_text=entity_text,
            case_sensitive=True
        )

        case_insensitive_occurrences = []

        if exact_occurrences:
            candidate_offsets = exact_occurrences
            match_method = "exact"
        else:
            case_insensitive_occurrences = find_all_text_occurrences(
                document_text=document_text,
                entity_text=entity_text,
                case_sensitive=False
            )

            candidate_offsets = case_insensitive_occurrences
            match_method = (
                "case_insensitive"
                if case_insensitive_occurrences
                else "not_found"
            )

        occurrence_count = len(candidate_offsets)

        if occurrence_count == 1:
            recovered_entity["start"] = (
                candidate_offsets[0]["start"]
            )
            recovered_entity["end"] = (
                candidate_offsets[0]["end"]
            )
            recovered_entity["document_text"] = (
                candidate_offsets[0]["document_text"]
            )
            recovered_entity["offset_status"] = (
                f"unique_{match_method}_match"
            )
            recovered_entity["candidate_offsets"] = (
                candidate_offsets
            )

            if match_method == "exact":
                summary["unique_exact_matches"] += 1
            else:
                summary[
                    "unique_case_insensitive_matches"
                ] += 1

        elif occurrence_count > 1:
            recovered_entity["start"] = None
            recovered_entity["end"] = None
            recovered_entity["document_text"] = None
            recovered_entity["offset_status"] = (
                f"multiple_{match_method}_matches"
            )
            recovered_entity["candidate_offsets"] = (
                candidate_offsets
            )

            if match_method == "exact":
                summary["multiple_exact_matches"] += 1
            else:
                summary[
                    "multiple_case_insensitive_matches"
                ] += 1

        else:
            recovered_entity["start"] = None
            recovered_entity["end"] = None
            recovered_entity["document_text"] = None
            recovered_entity["offset_status"] = "not_found"
            recovered_entity["candidate_offsets"] = []

            summary["not_found"] += 1

        recovered_entity["occurrence_count"] = (
            occurrence_count
        )

        recovered_entities.append(recovered_entity)

    recovered_output = {
        "entities": recovered_entities,
        "relations": parsed_output.get("relations", [])
    }

    return recovered_output, summary

In [67]:
pilot_output_with_offsets, offset_summary = (
    recover_entity_offsets(
        parsed_output=parsed_response,
        document_text=pilot_document["text"]
    )
)

print("Offset recovery summary:")

for key, value in offset_summary.items():
    print(f"- {key}: {value}")

Offset recovery summary:
- unique_exact_matches: 11
- multiple_exact_matches: 4
- unique_case_insensitive_matches: 2
- multiple_case_insensitive_matches: 0
- not_found: 0


In [68]:
for entity in pilot_output_with_offsets["entities"]:
    print(
        entity["id"],
        "|",
        repr(entity["text"]),
        "|",
        entity["offset_status"],
        "| start:",
        entity["start"],
        "| end:",
        entity["end"],
        "| occurrences:",
        entity["occurrence_count"]
    )

E1 | 'Congenital hypothyroidism' | unique_exact_match | start: 0 | end: 25 | occurrences: 1
E2 | 'sodium/iodide symporter protein' | unique_exact_match | start: 55 | end: 86 | occurrences: 1
E3 | 'iodide transport defect' | unique_case_insensitive_match | start: 99 | end: 122 | occurrences: 1
E4 | 'thyroid' | multiple_exact_matches | start: None | end: None | occurrences: 9
E5 | 'salivary glands' | multiple_exact_matches | start: None | end: None | occurrences: 3
E6 | 'gastric mucosa' | unique_exact_match | start: 399 | end: 413 | occurrences: 1
E7 | 'sodium/iodide symporter (NIS)' | unique_exact_match | start: 471 | end: 500 | occurrences: 1
E8 | 'hypothyroidism' | multiple_exact_matches | start: None | end: None | occurrences: 5
E9 | 'nodular goitre' | unique_exact_match | start: 634 | end: 648 | occurrences: 1
E10 | 'L-thyroxine' | unique_exact_match | start: 825 | end: 836 | occurrences: 1
E11 | 'thyroid radioiodide uptake' | unique_case_insensitive_match | start: 864 | end: 890 | 

In [69]:
offset_output_path = (
    pilot_output_dir /
    f'{pilot_document["document_id"]}_pilot_output_with_offsets.json'
)

offset_result = {
    "document_id": pilot_document["document_id"],
    "model": MODEL_NAME,
    "offset_summary": offset_summary,
    "output_with_offsets": pilot_output_with_offsets
}

with open(offset_output_path, "w", encoding="utf-8") as file:
    json.dump(
        offset_result,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Offset-enriched pilot output saved to:")
print(offset_output_path)

Offset-enriched pilot output saved to:
/content/drive/MyDrive/Capstone_Project/Stage_4_Outputs/Pilot_Outputs/14510914_pilot_output_with_offsets.json


### Stage 4.4E: Pilot Gold-Standard Comparison

This stage compares the first Qwen prediction with the BioRED gold-standard
annotations for the same development document.

The comparison will evaluate:

- predicted entities against gold entities;
- predicted relation types and entity pairs against gold relations;
- true positives, false positives and false negatives;
- precision, recall and F1-score;
- common LLM errors such as incorrect entity type, incorrect span,
  missing entity and unsupported relation.

This is still part of prompt-development testing. The final evaluation set
will not be used to modify the prompt.

In [70]:
pilot_document_id = str(pilot_document["document_id"])

raw_dev_documents = dev_data.get("documents", [])

original_pilot_document = next(
    (
        document
        for document in raw_dev_documents
        if str(document.get("id")) == pilot_document_id
    ),
    None
)

if original_pilot_document is None:
    raise ValueError(
        f"Document {pilot_document_id} was not found "
        "in dev_data['documents']."
    )

print("Original pilot document found.")
print("Document ID:", original_pilot_document.get("id"))
print("Document keys:", original_pilot_document.keys())

Original pilot document found.
Document ID: 14510914
Document keys: dict_keys(['id', 'passages', 'relations'])


In [71]:
raw_relations = original_pilot_document.get("relations", [])

corrected_gold_relations = []

for relation in raw_relations:
    infons = relation.get("infons", {})

    corrected_gold_relations.append({
        "relation_id": relation.get("id"),
        "type": infons.get("type"),
        "entity1_identifier": infons.get("entity1"),
        "entity2_identifier": infons.get("entity2"),
        "novel": infons.get("novel")
    })

print("Corrected gold relations:", len(corrected_gold_relations))

for relation in corrected_gold_relations:
    print(relation)

Corrected gold relations: 12
{'relation_id': 'R0', 'type': 'Positive_Correlation', 'entity1_identifier': 'c|DEL|1314_1328|', 'entity2_identifier': 'D003409', 'novel': 'Novel'}
{'relation_id': 'R1', 'type': 'Association', 'entity1_identifier': 'D050033', 'entity2_identifier': 'D007454', 'novel': 'No'}
{'relation_id': 'R2', 'type': 'Positive_Correlation', 'entity1_identifier': 'p|DEL|439_443|', 'entity2_identifier': 'C564766', 'novel': 'Novel'}
{'relation_id': 'R3', 'type': 'Positive_Correlation', 'entity1_identifier': 'p|DEL|439_443|', 'entity2_identifier': 'D003409', 'novel': 'Novel'}
{'relation_id': 'R4', 'type': 'Association', 'entity1_identifier': 'C564766', 'entity2_identifier': 'D007454', 'novel': 'No'}
{'relation_id': 'R5', 'type': 'Negative_Correlation', 'entity1_identifier': 'C564766', 'entity2_identifier': '6528', 'novel': 'No'}
{'relation_id': 'R6', 'type': 'Negative_Correlation', 'entity1_identifier': 'D013974', 'entity2_identifier': 'D007037', 'novel': 'No'}
{'relation_id':

In [72]:
pilot_document["gold_relations"] = corrected_gold_relations

print(
    "Pilot gold relations updated:",
    len(pilot_document["gold_relations"])
)

print("\nFirst corrected relation:")
print(pilot_document["gold_relations"][0])

Pilot gold relations updated: 12

First corrected relation:
{'relation_id': 'R0', 'type': 'Positive_Correlation', 'entity1_identifier': 'c|DEL|1314_1328|', 'entity2_identifier': 'D003409', 'novel': 'Novel'}


In [73]:
gold_identifiers = {
    entity["identifier"]
    for entity in pilot_document["gold_entities"]
}

unresolved_relation_identifiers = []

for relation in pilot_document["gold_relations"]:
    for field in [
        "entity1_identifier",
        "entity2_identifier"
    ]:
        identifier = relation.get(field)

        if identifier not in gold_identifiers:
            unresolved_relation_identifiers.append({
                "relation_id": relation["relation_id"],
                "field": field,
                "identifier": identifier
            })

print(
    "Unresolved relation identifiers:",
    len(unresolved_relation_identifiers)
)

if unresolved_relation_identifiers:
    for issue in unresolved_relation_identifiers:
        print(issue)

Unresolved relation identifiers: 0


In [74]:
gold_entity_keys = {
    (
        entity["start"],
        entity["end"],
        entity["type"]
    )
    for entity in pilot_document["gold_entities"]
}

print("Gold entity annotations:", len(gold_entity_keys))

Gold entity annotations: 37


In [75]:
predicted_entity_keys = set()
excluded_predicted_entities = []

for entity in pilot_output_with_offsets["entities"]:
    start = entity.get("start")
    end = entity.get("end")
    entity_type = entity.get("type")

    if start is not None and end is not None:
        predicted_entity_keys.add(
            (
                start,
                end,
                entity_type
            )
        )
    else:
        excluded_predicted_entities.append({
            "id": entity.get("id"),
            "text": entity.get("text"),
            "type": entity_type,
            "offset_status": entity.get("offset_status")
        })

print(
    "Predicted entities with resolved offsets:",
    len(predicted_entity_keys)
)

print(
    "Predicted entities excluded due to ambiguous offsets:",
    len(excluded_predicted_entities)
)

for entity in excluded_predicted_entities:
    print(entity)

Predicted entities with resolved offsets: 13
Predicted entities excluded due to ambiguous offsets: 4
{'id': 'E4', 'text': 'thyroid', 'type': 'OrganismTaxon', 'offset_status': 'multiple_exact_matches'}
{'id': 'E5', 'text': 'salivary glands', 'type': 'OrganismTaxon', 'offset_status': 'multiple_exact_matches'}
{'id': 'E8', 'text': 'hypothyroidism', 'type': 'DiseaseOrPhenotypicFeature', 'offset_status': 'multiple_exact_matches'}
{'id': 'E14', 'text': 'NIS gene', 'type': 'GeneOrGeneProduct', 'offset_status': 'multiple_exact_matches'}


In [76]:
true_positive_entities = (
    predicted_entity_keys & gold_entity_keys
)

false_positive_entities = (
    predicted_entity_keys - gold_entity_keys
)

false_negative_entities = (
    gold_entity_keys - predicted_entity_keys
)

entity_tp = len(true_positive_entities)
entity_fp = len(false_positive_entities)
entity_fn = len(false_negative_entities)

entity_precision = (
    entity_tp / (entity_tp + entity_fp)
    if (entity_tp + entity_fp) > 0
    else 0.0
)

entity_recall = (
    entity_tp / (entity_tp + entity_fn)
    if (entity_tp + entity_fn) > 0
    else 0.0
)

entity_f1 = (
    2 * entity_precision * entity_recall
    / (entity_precision + entity_recall)
    if (entity_precision + entity_recall) > 0
    else 0.0
)

print("Pilot strict NER evaluation")
print("---------------------------")
print("True positives:", entity_tp)
print("False positives:", entity_fp)
print("False negatives:", entity_fn)
print(f"Precision: {entity_precision:.4f}")
print(f"Recall:    {entity_recall:.4f}")
print(f"F1-score:  {entity_f1:.4f}")

Pilot strict NER evaluation
---------------------------
True positives: 4
False positives: 9
False negatives: 33
Precision: 0.3077
Recall:    0.1081
F1-score:  0.1600


In [77]:
print("False-positive entities:")
print()

for start, end, entity_type in sorted(
    false_positive_entities
):
    print({
        "text": pilot_document["text"][start:end],
        "start": start,
        "end": end,
        "predicted_type": entity_type
    })

False-positive entities:

{'text': 'sodium/iodide symporter protein', 'start': 55, 'end': 86, 'predicted_type': 'GeneOrGeneProduct'}
{'text': 'gastric mucosa', 'start': 399, 'end': 413, 'predicted_type': 'OrganismTaxon'}
{'text': 'sodium/iodide symporter (NIS)', 'start': 471, 'end': 500, 'predicted_type': 'GeneOrGeneProduct'}
{'text': 'Thyroid radioiodide uptake', 'start': 864, 'end': 890, 'predicted_type': 'ChemicalEntity'}
{'text': 'recombinant human TSH', 'start': 956, 'end': 977, 'predicted_type': 'ChemicalEntity'}
{'text': 'saliva to plasma I- ratio', 'start': 983, 'end': 1008, 'predicted_type': 'ChemicalEntity'}
{'text': '15 nucleotide (nt) deletion of the coding sequence (nt 1314 through nt 1328) and the insertion of 15 nt duplicating the first 15 nt of the adjacent intron', 'start': 1182, 'end': 1336, 'predicted_type': 'SequenceVariant'}
{'text': 'COS-7 cells', 'start': 1606, 'end': 1617, 'predicted_type': 'CellLine'}
{'text': 'mutant del-(439-443) NIS', 'start': 1659, 'end': 1

In [78]:
print("False-negative gold entities:")
print()

for start, end, entity_type in sorted(
    false_negative_entities
):
    print({
        "text": pilot_document["text"][start:end],
        "start": start,
        "end": end,
        "gold_type": entity_type
    })

False-negative gold entities:

{'text': 'sodium/iodide symporter', 'start': 55, 'end': 78, 'gold_type': 'GeneOrGeneProduct'}
{'text': 'ITD', 'start': 124, 'end': 127, 'gold_type': 'DiseaseOrPhenotypicFeature'}
{'text': 'inability of the thyroid', 'start': 168, 'end': 192, 'gold_type': 'DiseaseOrPhenotypicFeature'}
{'text': 'iodide', 'start': 208, 'end': 214, 'gold_type': 'ChemicalEntity'}
{'text': 'congenital hypothyroidism', 'start': 307, 'end': 332, 'gold_type': 'DiseaseOrPhenotypicFeature'}
{'text': 'sodium/iodide symporter', 'start': 471, 'end': 494, 'gold_type': 'GeneOrGeneProduct'}
{'text': 'NIS', 'start': 496, 'end': 499, 'gold_type': 'GeneOrGeneProduct'}
{'text': 'PATIENT', 'start': 502, 'end': 509, 'gold_type': 'OrganismTaxon'}
{'text': 'woman', 'start': 525, 'end': 530, 'gold_type': 'OrganismTaxon'}
{'text': 'hypothyroidism', 'start': 536, 'end': 550, 'gold_type': 'DiseaseOrPhenotypicFeature'}
{'text': 'ITD', 'start': 605, 'end': 608, 'gold_type': 'DiseaseOrPhenotypicFeature'

In [79]:
print("False-negative gold entities:")
print()

for start, end, entity_type in sorted(
    false_negative_entities
):
    print({
        "text": pilot_document["text"][start:end],
        "start": start,
        "end": end,
        "gold_type": entity_type
    })

False-negative gold entities:

{'text': 'sodium/iodide symporter', 'start': 55, 'end': 78, 'gold_type': 'GeneOrGeneProduct'}
{'text': 'ITD', 'start': 124, 'end': 127, 'gold_type': 'DiseaseOrPhenotypicFeature'}
{'text': 'inability of the thyroid', 'start': 168, 'end': 192, 'gold_type': 'DiseaseOrPhenotypicFeature'}
{'text': 'iodide', 'start': 208, 'end': 214, 'gold_type': 'ChemicalEntity'}
{'text': 'congenital hypothyroidism', 'start': 307, 'end': 332, 'gold_type': 'DiseaseOrPhenotypicFeature'}
{'text': 'sodium/iodide symporter', 'start': 471, 'end': 494, 'gold_type': 'GeneOrGeneProduct'}
{'text': 'NIS', 'start': 496, 'end': 499, 'gold_type': 'GeneOrGeneProduct'}
{'text': 'PATIENT', 'start': 502, 'end': 509, 'gold_type': 'OrganismTaxon'}
{'text': 'woman', 'start': 525, 'end': 530, 'gold_type': 'OrganismTaxon'}
{'text': 'hypothyroidism', 'start': 536, 'end': 550, 'gold_type': 'DiseaseOrPhenotypicFeature'}
{'text': 'ITD', 'start': 605, 'end': 608, 'gold_type': 'DiseaseOrPhenotypicFeature'

### Stage 4.4F: Controlled Prompt Revision

The first pilot achieved a strict mention-level NER F1-score of 0.1600.

The main errors were:

- extracting a broader phrase than the BioRED gold span;
- merging separate entities into one entity;
- missing repeated mentions and abbreviations;
- assigning anatomical locations to `OrganismTaxon`;
- assigning measurement phrases to `ChemicalEntity`.

A revised zero-shot prompt will now be tested on the same pilot document.

Only one controlled prompt revision will be made before testing the remaining
pilot documents.

In [80]:
def build_zero_shot_prompt_v2(document_text):
    entity_types_text = "\n".join(
        f'- "{entity_type}"'
        for entity_type in BIORED_ENTITY_TYPES
    )

    relation_types_text = "\n".join(
        f'- "{relation_type}"'
        for relation_type in BIORED_RELATION_TYPES
    )

    return f"""
You are a biomedical information extraction system.

Extract all BioRED biomedical entity mentions and relations from the document.

PERMITTED ENTITY TYPES:
{entity_types_text}

PERMITTED RELATION TYPES:
{relation_types_text}

ENTITY EXTRACTION RULES:

1. Extract every explicitly mentioned biomedical entity occurrence.
2. Treat repeated occurrences of the same entity as separate entity records.
3. Treat full names and abbreviations as separate mentions when both appear.
4. Copy entity text exactly from the document, preserving:
   - capitalisation;
   - punctuation;
   - hyphens;
   - symbols;
   - abbreviations.
5. Use the smallest complete span corresponding to the biomedical entity.
6. Do not include surrounding descriptive words when they are not part of the
   entity.
7. Do not combine two adjacent biomedical entities into one entity record.
8. Separate sequence variants from the genes or proteins they modify.
9. Assign every entity mention a unique ID: E1, E2, E3, and so on.
10. Use only the permitted BioRED entity types.

ENTITY TYPE GUIDANCE:

- GeneOrGeneProduct:
  genes, proteins, gene symbols and protein abbreviations.

- DiseaseOrPhenotypicFeature:
  diseases, disorders, symptoms, phenotypes and abnormal biological
  conditions.

- ChemicalEntity:
  drugs, chemical substances, ions and chemical compounds.
  Do not classify measurement phrases or biological processes as chemicals.

- SequenceVariant:
  deletions, insertions, substitutions, mutations and other sequence changes.
  Do not merge a variant with the associated gene or protein.

- CellLine:
  named experimental cell lines.
  Exclude generic words such as "cells" when the named cell line alone is the
  biomedical entity.

- OrganismTaxon:
  humans, patients, women, men, organisms, species and taxonomic groups.
  Do not classify organs, tissues, glands, anatomical structures or body
  locations as OrganismTaxon.

SPAN EXAMPLES:

- Extract "sodium/iodide symporter", not
  "sodium/iodide symporter protein", when "protein" is only descriptive.

- Extract "COS-7", not "COS-7 cells".

- From "mutant del-(439-443) NIS", extract:
  - "del-(439-443)" as SequenceVariant;
  - "NIS" as GeneOrGeneProduct.

- Do not extract phrases such as "thyroid radioiodide uptake" as a chemical.
  Extract only the actual annotated biomedical entity mention within it.

RELATION RULES:

1. Extract only relations explicitly supported by the document.
2. Use only the permitted BioRED relation types.
3. Relation head and tail values must reference valid predicted entity IDs.
4. Do not create relations involving entities that were not extracted.
5. Do not connect an entity to itself.
6. Do not infer relations from general biomedical knowledge.
7. Relations should connect the relevant entity mentions identified in the
   document.

OUTPUT RULES:

1. Return exactly one valid JSON object.
2. Do not use Markdown code fences.
3. Do not include explanations before or after the JSON.
4. Use exactly the keys shown below.
5. Return empty lists when no entities or relations are found.

REQUIRED OUTPUT STRUCTURE:

{{
  "entities": [
    {{
      "id": "E1",
      "text": "exact entity text from the document",
      "type": "one permitted BioRED entity type"
    }}
  ],
  "relations": [
    {{
      "head": "E1",
      "tail": "E2",
      "type": "one permitted BioRED relation type"
    }}
  ]
}}

DOCUMENT:
{document_text}
""".strip()

In [81]:
pilot_prompt_v2 = build_zero_shot_prompt_v2(
    document_text=pilot_document["text"]
)

print("Revised prompt length:", len(pilot_prompt_v2))
print()
print(pilot_prompt_v2[:2500])

Revised prompt length: 5532

You are a biomedical information extraction system.

Extract all BioRED biomedical entity mentions and relations from the document.

PERMITTED ENTITY TYPES:
- "CellLine"
- "ChemicalEntity"
- "DiseaseOrPhenotypicFeature"
- "GeneOrGeneProduct"
- "OrganismTaxon"
- "SequenceVariant"

PERMITTED RELATION TYPES:
- "Association"
- "Bind"
- "Comparison"
- "Conversion"
- "Cotreatment"
- "Drug_Interaction"
- "Negative_Correlation"
- "Positive_Correlation"

ENTITY EXTRACTION RULES:

1. Extract every explicitly mentioned biomedical entity occurrence.
2. Treat repeated occurrences of the same entity as separate entity records.
3. Treat full names and abbreviations as separate mentions when both appear.
4. Copy entity text exactly from the document, preserving:
   - capitalisation;
   - punctuation;
   - hyphens;
   - symbols;
   - abbreviations.
5. Use the smallest complete span corresponding to the biomedical entity.
6. Do not include surrounding descriptive words when 

In [82]:
messages_v2 = [
    {
        "role": "system",
        "content": (
            "You are a biomedical information extraction system. "
            "Follow all span and entity-type rules exactly. "
            "Return valid JSON only."
        )
    },
    {
        "role": "user",
        "content": pilot_prompt_v2
    }
]

formatted_prompt_v2 = tokenizer.apply_chat_template(
    messages_v2,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs_v2 = tokenizer(
    formatted_prompt_v2,
    return_tensors="pt"
).to(model.device)

input_token_count_v2 = model_inputs_v2["input_ids"].shape[1]

print("Revised formatted prompt length:", len(formatted_prompt_v2))
print("Revised input token count:", input_token_count_v2)

Revised formatted prompt length: 5734
Revised input token count: 1351


In [83]:
with torch.no_grad():
    generated_ids_v2 = model.generate(
        **model_inputs_v2,
        max_new_tokens=1600,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id
    )

new_token_ids_v2 = generated_ids_v2[
    :,
    model_inputs_v2["input_ids"].shape[1]:
]

raw_response_v2 = tokenizer.batch_decode(
    new_token_ids_v2,
    skip_special_tokens=True
)[0]

print(
    "Generated tokens:",
    new_token_ids_v2.shape[1]
)

print("\nRaw revised response:")
print(raw_response_v2)

Generated tokens: 915

Raw revised response:
{
  "entities": [
    {
      "id": "E1",
      "text": "congenital hypothyroidism",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
      "id": "E2",
      "text": "sodium/iodide symporter protein",
      "type": "GeneOrGeneProduct"
    },
    {
      "id": "E3",
      "text": "thyroid radioiodide uptake",
      "type": "ChemicalEntity"
    },
    {
      "id": "E4",
      "text": "thyroid",
      "type": "OrganismTaxon"
    },
    {
      "id": "E5",
      "text": "salivary glands",
      "type": "OrganismTaxon"
    },
    {
      "id": "E6",
      "text": "gastric mucosa",
      "type": "OrganismTaxon"
    },
    {
      "id": "E7",
      "text": "iodide transport defect",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
      "id": "E8",
      "text": "hypothyroidism",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
      "id": "E9",
      "text": "nodular goitre",
      "type": "DiseaseOrPhenotypicFeature"
    },


In [84]:
import json

try:
    parsed_response_v2 = json.loads(raw_response_v2)
    revised_json_valid = True
    revised_json_error = None

except json.JSONDecodeError as error:
    parsed_response_v2 = None
    revised_json_valid = False
    revised_json_error = str(error)

print("Valid JSON:", revised_json_valid)

if revised_json_error:
    print("JSON error:", revised_json_error)

if parsed_response_v2 is not None:
    print(
        "Predicted entities:",
        len(parsed_response_v2.get("entities", []))
    )
    print(
        "Predicted relations:",
        len(parsed_response_v2.get("relations", []))
    )

Valid JSON: True
Predicted entities: 15
Predicted relations: 14


In [85]:
revised_validation = validate_llm_extraction(
    parsed_output=parsed_response_v2,
    document_text=pilot_document["text"],
    permitted_entity_types=BIORED_ENTITY_TYPES,
    permitted_relation_types=BIORED_RELATION_TYPES
)

print("Structurally valid:", revised_validation["valid"])
print("Number of errors:", len(revised_validation["errors"]))
print("Number of warnings:", len(revised_validation["warnings"]))

if revised_validation["errors"]:
    print("\nValidation errors:")
    for error in revised_validation["errors"]:
        print("-", error)

if revised_validation["warnings"]:
    print("\nValidation warnings:")
    for warning in revised_validation["warnings"]:
        print("-", warning)

Structurally valid: False
Number of errors: 1
Number of warnings: 2

Validation errors:
- Entity E13: text not found in the document: '15 nt duplication'

Validation warnings:
- Entity E3: text found only with different capitalisation: 'thyroid radioiodide uptake'
- Entity E7: text found only with different capitalisation: 'iodide transport defect'


In [86]:
pilot_output_v2_with_offsets, offset_summary_v2 = (
    recover_entity_offsets(
        parsed_output=parsed_response_v2,
        document_text=pilot_document["text"]
    )
)

print("Revised offset recovery summary:")

for key, value in offset_summary_v2.items():
    print(f"- {key}: {value}")

Revised offset recovery summary:
- unique_exact_matches: 7
- multiple_exact_matches: 5
- unique_case_insensitive_matches: 2
- multiple_case_insensitive_matches: 0
- not_found: 1


In [87]:
for entity in pilot_output_v2_with_offsets["entities"]:
    print(
        entity["id"],
        "|",
        repr(entity["text"]),
        "|",
        entity["type"],
        "|",
        entity["offset_status"],
        "| start:",
        entity["start"],
        "| end:",
        entity["end"],
        "| occurrences:",
        entity["occurrence_count"]
    )

E1 | 'congenital hypothyroidism' | DiseaseOrPhenotypicFeature | multiple_exact_matches | start: None | end: None | occurrences: 2
E2 | 'sodium/iodide symporter protein' | GeneOrGeneProduct | unique_exact_match | start: 55 | end: 86 | occurrences: 1
E3 | 'thyroid radioiodide uptake' | ChemicalEntity | unique_case_insensitive_match | start: 864 | end: 890 | occurrences: 1
E4 | 'thyroid' | OrganismTaxon | multiple_exact_matches | start: None | end: None | occurrences: 9
E5 | 'salivary glands' | OrganismTaxon | multiple_exact_matches | start: None | end: None | occurrences: 3
E6 | 'gastric mucosa' | OrganismTaxon | unique_exact_match | start: 399 | end: 413 | occurrences: 1
E7 | 'iodide transport defect' | DiseaseOrPhenotypicFeature | unique_case_insensitive_match | start: 99 | end: 122 | occurrences: 1
E8 | 'hypothyroidism' | DiseaseOrPhenotypicFeature | multiple_exact_matches | start: None | end: None | occurrences: 5
E9 | 'nodular goitre' | DiseaseOrPhenotypicFeature | unique_exact_matc

In [88]:
predicted_entity_keys_v2 = set()
excluded_predicted_entities_v2 = []

for entity in pilot_output_v2_with_offsets["entities"]:
    start = entity.get("start")
    end = entity.get("end")
    entity_type = entity.get("type")

    if start is not None and end is not None:
        predicted_entity_keys_v2.add(
            (start, end, entity_type)
        )
    else:
        excluded_predicted_entities_v2.append({
            "id": entity.get("id"),
            "text": entity.get("text"),
            "type": entity_type,
            "offset_status": entity.get("offset_status")
        })

true_positive_entities_v2 = (
    predicted_entity_keys_v2 & gold_entity_keys
)

false_positive_entities_v2 = (
    predicted_entity_keys_v2 - gold_entity_keys
)

false_negative_entities_v2 = (
    gold_entity_keys - predicted_entity_keys_v2
)

entity_tp_v2 = len(true_positive_entities_v2)
entity_fp_v2 = len(false_positive_entities_v2)
entity_fn_v2 = len(false_negative_entities_v2)

entity_precision_v2 = (
    entity_tp_v2 / (entity_tp_v2 + entity_fp_v2)
    if (entity_tp_v2 + entity_fp_v2) > 0
    else 0.0
)

entity_recall_v2 = (
    entity_tp_v2 / (entity_tp_v2 + entity_fn_v2)
    if (entity_tp_v2 + entity_fn_v2) > 0
    else 0.0
)

entity_f1_v2 = (
    2 * entity_precision_v2 * entity_recall_v2
    / (entity_precision_v2 + entity_recall_v2)
    if (entity_precision_v2 + entity_recall_v2) > 0
    else 0.0
)

print("Revised pilot strict NER evaluation")
print("-----------------------------------")
print("True positives:", entity_tp_v2)
print("False positives:", entity_fp_v2)
print("False negatives:", entity_fn_v2)
print(f"Precision: {entity_precision_v2:.4f}")
print(f"Recall:    {entity_recall_v2:.4f}")
print(f"F1-score:  {entity_f1_v2:.4f}")

print("\nOriginal versus revised")
print("-----------------------")
print(f"Original F1:  {entity_f1:.4f}")
print(f"Revised F1:   {entity_f1_v2:.4f}")
print(f"F1 change:    {entity_f1_v2 - entity_f1:+.4f}")

Revised pilot strict NER evaluation
-----------------------------------
True positives: 4
False positives: 5
False negatives: 33
Precision: 0.4444
Recall:    0.1081
F1-score:  0.1739

Original versus revised
-----------------------
Original F1:  0.1600
Revised F1:   0.1739
F1 change:    +0.0139


### Stage 4.4G: Frozen-Prompt Pilot Evaluation

Prompt version 2 is now frozen.

The same revised prompt and deterministic generation settings will be applied
to the remaining two development pilot documents without further modification.

This checks whether the small improvement observed on the first pilot document
generalises to other documents.

In [89]:
remaining_pilot_documents = prompt_development_sample[1:]

print("Remaining pilot documents:", len(remaining_pilot_documents))

for document in remaining_pilot_documents:
    print(
        "Document ID:",
        document["document_id"],
        "| Text length:",
        len(document["text"]),
        "| Gold entities:",
        len(document["gold_entities"]),
        "| Gold relations:",
        len(document["gold_relations"])
    )

Remaining pilot documents: 2
Document ID: 15096016 | Text length: 1362 | Gold entities: 14 | Gold relations: 1
Document ID: 16152606 | Text length: 1802 | Gold entities: 40 | Gold relations: 10


In [90]:
def generate_with_frozen_prompt(document):
    prompt = build_zero_shot_prompt_v2(
        document_text=document["text"]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a biomedical information extraction system. "
                "Follow all span and entity-type rules exactly. "
                "Return valid JSON only."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1600,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    new_token_ids = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:
    ]

    raw_response = tokenizer.batch_decode(
        new_token_ids,
        skip_special_tokens=True
    )[0]

    try:
        parsed_response = json.loads(raw_response)
        json_valid = True
        json_error = None

    except json.JSONDecodeError as error:
        parsed_response = None
        json_valid = False
        json_error = str(error)

    return {
        "document_id": document["document_id"],
        "prompt": prompt,
        "input_token_count": model_inputs["input_ids"].shape[1],
        "generated_token_count": new_token_ids.shape[1],
        "raw_response": raw_response,
        "parsed_response": parsed_response,
        "json_valid": json_valid,
        "json_error": json_error
    }

In [91]:
pilot_result_2 = generate_with_frozen_prompt(
    remaining_pilot_documents[0]
)

print("Document ID:", pilot_result_2["document_id"])
print("Input tokens:", pilot_result_2["input_token_count"])
print("Generated tokens:", pilot_result_2["generated_token_count"])
print("Valid JSON:", pilot_result_2["json_valid"])

if pilot_result_2["json_error"]:
    print("JSON error:", pilot_result_2["json_error"])

if pilot_result_2["parsed_response"] is not None:
    print(
        "Predicted entities:",
        len(
            pilot_result_2["parsed_response"].get(
                "entities",
                []
            )
        )
    )

    print(
        "Predicted relations:",
        len(
            pilot_result_2["parsed_response"].get(
                "relations",
                []
            )
        )
    )

Document ID: 15096016
Input tokens: 1172
Generated tokens: 561
Valid JSON: True
Predicted entities: 16
Predicted relations: 0


In [92]:
pilot_document_2 = remaining_pilot_documents[0]
parsed_response_2 = pilot_result_2["parsed_response"]

validation_2 = validate_llm_extraction(
    parsed_output=parsed_response_2,
    document_text=pilot_document_2["text"],
    permitted_entity_types=BIORED_ENTITY_TYPES,
    permitted_relation_types=BIORED_RELATION_TYPES
)

print("Structurally valid:", validation_2["valid"])
print("Number of errors:", len(validation_2["errors"]))
print("Number of warnings:", len(validation_2["warnings"]))

if validation_2["errors"]:
    print("\nValidation errors:")
    for error in validation_2["errors"]:
        print("-", error)

if validation_2["warnings"]:
    print("\nValidation warnings:")
    for warning in validation_2["warnings"]:
        print("-", warning)

Structurally valid: False
Number of errors: 7
Number of warnings: 0

Validation errors:
- Entity E2 has invalid type: 'Procedure'
- Entity E3 has invalid type: 'Procedure'
- Entity E4 has invalid type: 'AnatomicalStructure'
- Entity E5 has invalid type: 'Procedure'
- Entity E13 has invalid type: 'AnatomicalStructure'
- Entity E15 has invalid type: 'Device'
- Entity E16 has invalid type: 'Device'


In [93]:
pilot_document_2 = remaining_pilot_documents[0]
parsed_response_2 = pilot_result_2["parsed_response"]

print("Document ID:", pilot_document_2["document_id"])
print("Parsed response available:", parsed_response_2 is not None)

Document ID: 15096016
Parsed response available: True


In [94]:
pilot_output_2_with_offsets, offset_summary_2 = (
    recover_entity_offsets(
        parsed_output=parsed_response_2,
        document_text=pilot_document_2["text"]
    )
)

print("Pilot 2 offset recovery summary:")

for key, value in offset_summary_2.items():
    print(f"- {key}: {value}")

Pilot 2 offset recovery summary:
- unique_exact_matches: 12
- multiple_exact_matches: 4
- unique_case_insensitive_matches: 0
- multiple_case_insensitive_matches: 0
- not_found: 0


In [95]:
gold_entity_keys_2 = {
    (
        entity["start"],
        entity["end"],
        entity["type"]
    )
    for entity in pilot_document_2["gold_entities"]
}

predicted_entity_keys_2 = set()
excluded_entities_2 = []

for entity in pilot_output_2_with_offsets["entities"]:
    start = entity.get("start")
    end = entity.get("end")
    entity_type = entity.get("type")

    if start is not None and end is not None:
        predicted_entity_keys_2.add(
            (start, end, entity_type)
        )
    else:
        excluded_entities_2.append({
            "id": entity.get("id"),
            "text": entity.get("text"),
            "type": entity_type,
            "offset_status": entity.get("offset_status")
        })

true_positive_entities_2 = (
    predicted_entity_keys_2 & gold_entity_keys_2
)

false_positive_entities_2 = (
    predicted_entity_keys_2 - gold_entity_keys_2
)

false_negative_entities_2 = (
    gold_entity_keys_2 - predicted_entity_keys_2
)

tp_2 = len(true_positive_entities_2)
fp_2 = len(false_positive_entities_2)
fn_2 = len(false_negative_entities_2)

precision_2 = (
    tp_2 / (tp_2 + fp_2)
    if (tp_2 + fp_2) > 0
    else 0.0
)

recall_2 = (
    tp_2 / (tp_2 + fn_2)
    if (tp_2 + fn_2) > 0
    else 0.0
)

f1_2 = (
    2 * precision_2 * recall_2
    / (precision_2 + recall_2)
    if (precision_2 + recall_2) > 0
    else 0.0
)

print("Pilot 2 strict NER evaluation")
print("-----------------------------")
print("Gold entities:", len(gold_entity_keys_2))
print("Resolved predicted entities:", len(predicted_entity_keys_2))
print("Excluded ambiguous entities:", len(excluded_entities_2))
print("True positives:", tp_2)
print("False positives:", fp_2)
print("False negatives:", fn_2)
print(f"Precision: {precision_2:.4f}")
print(f"Recall:    {recall_2:.4f}")
print(f"F1-score:  {f1_2:.4f}")

Pilot 2 strict NER evaluation
-----------------------------
Gold entities: 14
Resolved predicted entities: 12
Excluded ambiguous entities: 4
True positives: 1
False positives: 11
False negatives: 13
Precision: 0.0833
Recall:    0.0714
F1-score:  0.0769


In [97]:
pilot_result_3 = generate_with_frozen_prompt(
    remaining_pilot_documents[1]
)

print("Document ID:", pilot_result_3["document_id"])
print("Input tokens:", pilot_result_3["input_token_count"])
print("Generated tokens:", pilot_result_3["generated_token_count"])
print("Valid JSON:", pilot_result_3["json_valid"])

if pilot_result_3["json_error"]:
    print("JSON error:", pilot_result_3["json_error"])

if pilot_result_3["parsed_response"] is not None:
    print(
        "Predicted entities:",
        len(
            pilot_result_3["parsed_response"].get(
                "entities",
                []
            )
        )
    )

    print(
        "Predicted relations:",
        len(
            pilot_result_3["parsed_response"].get(
                "relations",
                []
            )
        )
    )

Document ID: 16152606
Input tokens: 1319
Generated tokens: 1295
Valid JSON: True
Predicted entities: 16
Predicted relations: 25


In [98]:
pilot_document_3 = remaining_pilot_documents[1]
parsed_response_3 = pilot_result_3["parsed_response"]

print("Document ID:", pilot_document_3["document_id"])
print("Parsed response available:", parsed_response_3 is not None)

Document ID: 16152606
Parsed response available: True


In [99]:
validation_3 = validate_llm_extraction(
    parsed_output=parsed_response_3,
    document_text=pilot_document_3["text"],
    permitted_entity_types=BIORED_ENTITY_TYPES,
    permitted_relation_types=BIORED_RELATION_TYPES
)

print("Structurally valid:", validation_3["valid"])
print("Number of errors:", len(validation_3["errors"]))
print("Number of warnings:", len(validation_3["warnings"]))

if validation_3["errors"]:
    print("\nValidation errors:")
    for error in validation_3["errors"]:
        print("-", error)

if validation_3["warnings"]:
    print("\nValidation warnings:")
    for warning in validation_3["warnings"]:
        print("-", warning)

Structurally valid: False
Number of errors: 3
Number of warnings: 0

Validation errors:
- Entity E10: text not found in the document: 'children with ALL'
- Relation 19 connects entity 'E15' to itself.
- Relation 20 connects entity 'E16' to itself.


In [100]:
pilot_output_3_with_offsets, offset_summary_3 = (
    recover_entity_offsets(
        parsed_output=parsed_response_3,
        document_text=pilot_document_3["text"]
    )
)

print("Pilot 3 offset recovery summary:")

for key, value in offset_summary_3.items():
    print(f"- {key}: {value}")

Pilot 3 offset recovery summary:
- unique_exact_matches: 10
- multiple_exact_matches: 5
- unique_case_insensitive_matches: 0
- multiple_case_insensitive_matches: 0
- not_found: 1


In [101]:
gold_entity_keys_3 = {
    (
        entity["start"],
        entity["end"],
        entity["type"]
    )
    for entity in pilot_document_3["gold_entities"]
}

predicted_entity_keys_3 = set()
excluded_entities_3 = []

for entity in pilot_output_3_with_offsets["entities"]:
    start = entity.get("start")
    end = entity.get("end")
    entity_type = entity.get("type")

    if start is not None and end is not None:
        predicted_entity_keys_3.add(
            (start, end, entity_type)
        )
    else:
        excluded_entities_3.append({
            "id": entity.get("id"),
            "text": entity.get("text"),
            "type": entity_type,
            "offset_status": entity.get("offset_status")
        })

true_positive_entities_3 = (
    predicted_entity_keys_3 & gold_entity_keys_3
)

false_positive_entities_3 = (
    predicted_entity_keys_3 - gold_entity_keys_3
)

false_negative_entities_3 = (
    gold_entity_keys_3 - predicted_entity_keys_3
)

tp_3 = len(true_positive_entities_3)
fp_3 = len(false_positive_entities_3)
fn_3 = len(false_negative_entities_3)

precision_3 = (
    tp_3 / (tp_3 + fp_3)
    if (tp_3 + fp_3) > 0
    else 0.0
)

recall_3 = (
    tp_3 / (tp_3 + fn_3)
    if (tp_3 + fn_3) > 0
    else 0.0
)

f1_3 = (
    2 * precision_3 * recall_3
    / (precision_3 + recall_3)
    if (precision_3 + recall_3) > 0
    else 0.0
)

print("Pilot 3 strict NER evaluation")
print("-----------------------------")
print("Gold entities:", len(gold_entity_keys_3))
print("Resolved predicted entities:", len(predicted_entity_keys_3))
print("Excluded ambiguous entities:", len(excluded_entities_3))
print("True positives:", tp_3)
print("False positives:", fp_3)
print("False negatives:", fn_3)
print(f"Precision: {precision_3:.4f}")
print(f"Recall:    {recall_3:.4f}")
print(f"F1-score:  {f1_3:.4f}")

Pilot 3 strict NER evaluation
-----------------------------
Gold entities: 40
Resolved predicted entities: 10
Excluded ambiguous entities: 6
True positives: 0
False positives: 10
False negatives: 40
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000
